# Presentacion del Sistema

Este notebook resume el problema, la solucion, lo que aprendio el bandit y como recomendar un canal para contextos vistos y no vistos.


## Análisis general del proyecto

El EDA mostro que no existe un solo canal ganador para todo. Más bien, el rendimiento depende bastante de la combinacion entre audiencia y objetivo de campana. Por eso la solucion final tiene mas sentido si se plantea como aprendizaje por contexto, y no como una recomendacion unica para todos los casos.


In [6]:
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.evaluacion.experimentos import ejecutar_flujo_completo


## Análisis de los resultados generales

Con esta ejecucion se entrenan `36` contextos. La accuracy del mejor canal queda en `0.75`, el ROI promedio del bandit queda cerca del optimo historico y el regret promedio se mantiene bajo. Para una primera version del sistema, eso ya da una senal bastante positiva.


In [7]:
# Inicializamos y ejecutamos el flujo completo del sistema de recomendación
resultado = ejecutar_flujo_completo(ROOT / 'social_media_ads_filtered.csv')

print('Estructura general del diccionario resultado:')
pprint({clave: type(valor).__name__ for clave, valor in resultado.items()})

print('\nResumen final del sistema:')
pprint(resultado['metricas_finales'])


Entrenando contexto: Women 45-60 | Brand Awareness
Entrenando contexto: Men 25-34 | Product Launch
Entrenando contexto: Men 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Market Expansion
Entrenando contexto: Women 25-34 | Increase Sales
Entrenando contexto: Women 35-44 | Brand Awareness
Entrenando contexto: Men 25-34 | Increase Sales
Entrenando contexto: Women 45-60 | Product Launch
Entrenando contexto: Women 35-44 | Market Expansion
Entrenando contexto: Men 45-60 | Market Expansion
Entrenando contexto: Women 18-24 | Product Launch
Entrenando contexto: Women 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Increase Sales
Entrenando contexto: Men 45-60 | Increase Sales
Entrenando contexto: Women 35-44 | Product Launch
Entrenando contexto: Women 25-34 | Brand Awareness
Entrenando contexto: Men 45-60 | Brand Awareness
Entrenando contexto: Men 35-44 | Brand Awareness
Entrenando contexto: Men 18-24 | Increase Sales
Entrenando contexto: Women 25-34 | Product Launch
Entrena

## Ejemplos de cold start sin histórico

Aca se ve algo importante del sistema: no solo puede responder para contextos conocidos, sino que tambien mantiene una salida razonable cuando aparece una audiencia nueva, un objetivo nuevo o ambos. En esos casos usa la estrategia de fallback definida en el proyecto.


In [8]:
print('Recomendacion para contexto conocido:')
pprint(resultado['recomendacion_ejemplo'])

print('\nCold start por audiencia nueva:')
pprint(resultado['cold_start_audiencia_nueva'])

print('\nCold start por objetivo nuevo:')
pprint(resultado['cold_start_objetivo_nuevo'])

print('\nCold start total:')
pprint(resultado['cold_start_total'])


Recomendacion para contexto conocido:
{'channel': 'Twitter',
 'contexto': 'Women 45-60 | Market Expansion',
 'detalle': [{'alpha': 77.0,
              'beta': 24.0,
              'canal': 'Twitter',
              'impresiones': 99,
              'muestra_esperada': 0.7623762376237624,
              'recompensa_promedio': 0.619166666666667,
              'roi_promedio_observado': 4.953333333333336},
             {'alpha': 6.0,
              'beta': 6.0,
              'canal': 'Instagram',
              'impresiones': 10,
              'muestra_esperada': 0.5,
              'recompensa_promedio': 0.5077499999999999,
              'roi_promedio_observado': 4.061999999999999},
             {'alpha': 1.0,
              'beta': 4.0,
              'canal': 'Facebook',
              'impresiones': 3,
              'muestra_esperada': 0.2,
              'recompensa_promedio': 0.38833333333333336,
              'roi_promedio_observado': 3.106666666666667},
             {'alpha': 1.0,
           

## Politíca aprendida

Esta tabla resume, por contexto, cual es el canal final que el agente termino recomendando. Ya no estamos viendo cada brazo por separado, sino la politica final que deja el sistema.

Es decir, la decisión que recomienda el sistema que se debe de tomar para cada contexto. 

Que canal se debe usar


In [ ]:
resultado['politica_aprendida'].head(20)

,Contexto,canal_recomendado
0,All Ages | Brand Awareness,Twitter
1,All Ages | Increase Sales,Twitter
2,All Ages | Market Expansion,Facebook
3,All Ages | Product Launch,Twitter
4,Men 18-24 | Brand Awareness,Facebook
5,Men 18-24 | Increase Sales,Instagram
6,Men 18-24 | Market Expansion,Twitter
7,Men 18-24 | Product Launch,Instagram
8,Men 25-34 | Brand Awareness,Facebook
9,Men 25-34 | Increase Sales,Instagram


## Evaluación del canal recomendado

Esta tabla junta la recomendacion del bandit con el mejor canal historico observado. La idea aca es ver si el sistema aprende el mismo canal ganador o, por lo menos, uno con un ROI bastante parecido. Eso nos permite evaluar no solo acierto exacto, sino tambien cercania en rendimiento.


In [10]:
resultado['evaluacion'].head(20)


,Contexto,canal_recomendado_bandit,mejor_canal_historico,acierto_mejor_canal,roi_promedio_bandit,roi_promedio_optimo,regret_aproximado
0,Women 45-60 | Brand Awareness,Facebook,Facebook,1,4.637473,4.584400,0.000000
1,Men 25-34 | Product Launch,Instagram,Instagram,1,4.060370,3.954828,0.000000
2,Men 18-24 | Brand Awareness,Facebook,Facebook,1,4.124737,4.187500,0.062763
3,Men 35-44 | Market Expansion,Instagram,Instagram,1,4.069677,3.981667,0.000000
4,Women 25-34 | Increase Sales,Instagram,Instagram,1,4.500357,4.499091,0.000000
5,Women 35-44 | Brand Awareness,Twitter,Twitter,1,4.698088,4.631538,0.000000
6,Men 25-34 | Increase Sales,Instagram,Instagram,1,4.361176,4.347586,0.000000
7,Women 45-60 | Product Launch,Twitter,Twitter,1,4.513297,4.500938,0.000000
8,Women 35-44 | Market Expansion,Instagram,Instagram,1,4.549500,4.455429,0.000000
9,Men 45-60 | Market Expansion,Facebook,Facebook,1,4.268644,4.247407,0.000000


## Qué aprendio cada brazo?

Acá bajamos un nivel más. En vez de ver solo el canal final, vemos las estadisticas internas por brazo dentro de cada contexto: impresiones, recompensa promedio, ROI promedio observado y los parametros bayesianos del Thompson Sampling. Esto ayuda bastante para explicar por que el sistema termina favoreciendo ciertos canales.


In [11]:
resultado['tabla_politica_brazos'].head(20)


,canal,impresiones,recompensa_promedio,roi_promedio_observado,alpha,beta,muestra_esperada,Contexto
0,Facebook,91,0.579684,4.637473,57.0,36.0,0.612903,Women 45-60 | Brand Awareness
1,Twitter,15,0.452417,3.619333,7.0,10.0,0.411765,Women 45-60 | Brand Awareness
2,Instagram,6,0.396875,3.175000,2.0,6.0,0.250000,Women 45-60 | Brand Awareness
3,Pinterest,5,0.081687,0.653500,1.0,6.0,0.142857,Women 45-60 | Brand Awareness
4,Instagram,54,0.507546,4.060370,26.0,30.0,0.464286,Men 25-34 | Product Launch
5,Twitter,25,0.456250,3.650000,12.0,15.0,0.444444,Men 25-34 | Product Launch
6,Facebook,47,0.419229,3.353830,21.0,28.0,0.428571,Men 25-34 | Product Launch
7,Pinterest,7,0.053880,0.431036,1.0,8.0,0.111111,Men 25-34 | Product Launch
8,Facebook,38,0.515592,4.124737,20.0,20.0,0.500000,Men 18-24 | Brand Awareness
9,Instagram,46,0.506522,4.052174,23.0,25.0,0.479167,Men 18-24 | Brand Awareness


## Comparación contextual vs global

Esta tabla ayuda mucho para la presentacion porque deja ver, por contexto, si el agente contextual o el global queda mejor. En varios casos el contextual supera al global, lo cual es coherente con lo que ya habiamos visto en el analisis exploratorio. La columna `mejor_agente` resume justo eso y hace mas facil defender por que conviene modelar este problema por combinacion de audiencia y objetivo.


In [12]:
resultado['comparacion_contextual_vs_global'].head(20)


,Contexto,canal_contextual,recompensa_promedio_contextual,roi_promedio_contextual,canal_global,recompensa_promedio_global,roi_promedio_global,coinciden,mejor_agente
0,Women 45-60 | Brand Awareness,Facebook,0.579684,4.637473,Twitter,0.497152,3.977217,0,contextual
1,Men 25-34 | Product Launch,Instagram,0.507546,4.060370,Twitter,0.497152,3.977217,0,contextual
2,Men 18-24 | Brand Awareness,Facebook,0.515592,4.124737,Twitter,0.497152,3.977217,0,contextual
3,Men 35-44 | Market Expansion,Instagram,0.508710,4.069677,Twitter,0.497152,3.977217,0,contextual
4,Women 25-34 | Increase Sales,Instagram,0.562545,4.500357,Twitter,0.497152,3.977217,0,contextual
5,Women 35-44 | Brand Awareness,Twitter,0.587261,4.698088,Twitter,0.497152,3.977217,1,contextual
6,Men 25-34 | Increase Sales,Instagram,0.545147,4.361176,Twitter,0.497152,3.977217,0,contextual
7,Women 45-60 | Product Launch,Twitter,0.564162,4.513297,Twitter,0.497152,3.977217,1,contextual
8,Women 35-44 | Market Expansion,Instagram,0.568688,4.549500,Twitter,0.497152,3.977217,0,contextual
9,Men 45-60 | Market Expansion,Facebook,0.533581,4.268644,Twitter,0.497152,3.977217,0,contextual


In [15]:
# Análisis final
resumen_presentacion = {
    'n_contextos': resultado['metricas_finales']['n_contextos'],
    'accuracy_mejor_canal': resultado['metricas_finales']['accuracy_mejor_canal'],
    'roi_promedio_bandit': resultado['metricas_finales']['roi_promedio_bandit'],
    'roi_promedio_optimo': resultado['metricas_finales']['roi_promedio_optimo'],
    'regret_aproximado_promedio': resultado['metricas_finales']['regret_aproximado_promedio']
}

print('Resumen ejecutivo del sistema:')
pprint(resumen_presentacion)


Resumen ejecutivo del sistema:
{'accuracy_mejor_canal': 0.75,
 'n_contextos': 36,
 'regret_aproximado_promedio': 0.10826605395884409,
 'roi_promedio_bandit': 4.246731149151057,
 'roi_promedio_optimo': 4.323162075092861}


## Conclusión final

La parte mas importante de esta solucion no es solo que recomiende un canal, sino que lo haga respetando la heterogeneidad del problema. Ese fue el hallazgo mas claro del analisis inicial.

En conjunto, los resultados muestran tres cosas:

- si vale la pena trabajar por contexto
- Thompson Sampling logra una politica bastante competitiva
- el sistema tambien responde de forma razonable en cold start

Entonces, mas que una recomendacion fija, lo que estamos construyendo aca es una politica adaptable para decidir que canal conviene segun el tipo de audiencia y el objetivo de campana.
